In [61]:
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
documents = loader.load()
docs = documents[0]

In [76]:
from pydantic import BaseModel, Field
page_content = docs.page_content[:10000]
class Overview(BaseModel):
    """Overview of a content"""
    summary: str = Field(description = "Provide the brief and simple summary of the content")
    language: str = Field(description= "The language in which the content is written")
    keywords: str = Field(description="Provide keywords relevant to the content")

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()
google_api_key = os.getenv('GOOGLE_API_KEY')
chat = ChatGoogleGenerativeAI(model = 'gemini-2.0-flash', temperature = 0.0)
model = chat.bind(functions = [Overview])


In [84]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.output_parsers import JsonOutputToolsParser
prompt = ChatPromptTemplate.from_messages([("system", "Extract the asked information from the content, if not explicitly provided do not guess. Extract partial info. Always use Overview tool to extract the info."), ("human", "{content}")] )

chain = prompt | model | JsonOutputToolsParser()

In [85]:
res = chain.invoke({'content': page_content})
print(res)

[{'args': {'language': 'english', 'summary': 'This article discusses LLM-powered autonomous agents, focusing on their architecture, including planning, memory, and tool use. It also covers task decomposition, self-reflection, and provides case studies and examples.', 'keywords': 'LLM, Autonomous Agents, Planning, Memory, Tool Use, Task Decomposition, Self-Reflection'}, 'type': 'Overview'}]


In [99]:
print(model.invoke(f"Extract the relevant information, if not explicitly provided do not guess. Extract partial info. Use Overview tool to extract info. The content from where you need to extract info is: {page_content}"))

content='' additional_kwargs={'function_call': {'name': 'Overview', 'arguments': '{"language": "english", "summary": "This article provides an overview of LLM-powered autonomous agents, focusing on their key components: planning, memory, and tool use. It discusses techniques like task decomposition, self-reflection, and different memory types. It also touches upon case studies and challenges in building these agents.", "keywords": "LLM, autonomous agents, planning, memory, tool use, self-reflection"}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []} id='run--c4e07bb1-9831-4e56-9958-ec685e835c08-0' tool_calls=[{'name': 'Overview', 'args': {'language': 'english', 'summary': 'This article provides an overview of LLM-powered autonomous agents, focusing on their key components: planning, memory, and tool use. It discusses techniques like task decomposition, self-reflection, and

In [107]:
from typing import Optional, List
class Paper(BaseModel):
    """Information about papers mentioned"""
    title: str = Field(description="What is the title of the page?")
    author: Optional[str] = Field(description="Who is the author of the page.")

class Info(BaseModel):
    """Information to extract out"""
    papers: List[Paper] = Field(description="The collection of pages")

func = [Info]
model1 = chat.bind(functions = func)
prompt_1 = ChatPromptTemplate.from_messages([
    ("system", "Extract the info from each pages which are asked from the content provided. if there isn't any info provided, leave it and Extract Partially. Always use Info Tool to take out the asked infos from the collection of pages."),
    ("human", "{context}")
])
chain_1 = prompt_1 | model1 | JsonOutputToolsParser()

In [108]:
print(chain_1.invoke({'context': page_content}))

[{'args': {'papers': [{'title': 'LLM Powered Autonomous Agents', 'author': 'Lilian Weng'}]}, 'type': 'Info'}]
